<a href="https://colab.research.google.com/github/a08037/-/blob/main/%D0%90%D1%82%D1%80%D0%B8%D0%B1%D1%83%D1%86%D0%B8%D0%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Настраиваем среду
import pandas as pd

pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

# Читаем файл с исходными данными о пользователях, активности и дате
df = pd.read_csv('https://gist.githubusercontent.com/andron23/9dc0200c84a52fcca6eb683360d2c02e/raw/f077693e4ceea5807aa6b3bfedcb553c5357a7fa/data.csv')

# Смотрим структуру датафрейма
df.head()

,email,created_at,tag
0,Студент 1,2024-01-25 19:30:41.905,Активность 1
1,Студент 1,2024-02-08 08:00:52.891,Активность 2
2,Студент 1,2024-02-21 18:28:12.031,Активность 3
3,Студент 1,2024-02-21 18:28:12.040,Активность 3
4,Студент 1,2024-03-10 10:02:02.804,Активность 4


In [ ]:
# Добавляем столбец rn, чтобы последняя активность каждого пользователя была помечена цифрой "1"
df['rn'] = (
    df
    .sort_values(['email','created_at'], ascending=[True, False])
    .groupby(['email'])
    .cumcount() + 1
)

# Берем только последние активности по каждому пользователю и выводим количество раз, когда это привело к покупке
# Т.е. выводим активности по стандартной last-click атрибуции
(
    df
     [df['rn'] == 1]
    .groupby('tag')
    ['rn']
    .count()
    .reset_index()
    .sort_values('rn', ascending=False)
)

,tag,rn
5,Активность 50,21
0,Активность 1,2
1,Активность 19,2
3,Активность 29,2
2,Активность 21,1
4,Активность 49,1


In [ ]:
# Переводим поле created_at в формат даты-времени
df['created_at'] = pd.to_datetime(df['created_at'])

# Для каждой активности каждого пользоватя смотрим - сколько дней прошло с момента последней активности (т.е. насколько давно эта активность была)
df['diff'] = (
    df
    .apply(lambda x: df[df['email'] == x['email']]['created_at'].max() - x['created_at'], axis=1)
    .dt
    .days
)

# Строим кастомную атрибуцию - назначаем веса
# Если меньше года: 1 - разница между текущей и последней активностью / 365. И еще возводим в 4 степень, для усиления
# Если больше года: 0.
df['diff2'] = df.apply(lambda x: (1 - x['diff'] / 365 if x['diff'] <= 365 else 0) ** 4, axis=1)

In [ ]:
# Выводим все активности, где значение веса с такой атрибуцией получилось не слишком маленьким и сортируем от самых важных к самым неважным
df2 = (
    df
    .groupby(['tag'])
    ['diff2']
    .sum()
    .reset_index()
    .sort_values('diff2', ascending=False)
)

df2[df2['diff2'] > 1e-3]

,tag,diff2
45,Активность 50,28.000000
0,Активность 1,20.319988
10,Активность 19,18.794600
21,Активность 29,15.069557
19,Активность 27,9.509223
31,Активность 38,8.740386
27,Активность 34,6.643805
9,Активность 18,6.414921
49,Активность 9,5.356869
7,Активность 16,3.727838
